In [ ]:
# ============================================================
# CELLULE 0 — A2 Topics : imports, style, palette, paths
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# ── Paths ────────────────────────────────────────────────────
BASE = Path(".")
DATA_A2 = BASE / "data"
DATA = Path("..") / "data"
DATA_A1_OUT = Path("..") / "A1_temporal" / "outputs"
OUT = BASE / "outputs"
OUT.mkdir(exist_ok=True)

# ── Palette candidats ────────────────────────────────────────
COLORS = {
    "Brossat": "#E63946",
    "Chikirou": "#C1121F",
    "Belliard": "#2D6A4F",
    "Gregoire": "#E07A5F",
    "Bournazel": "#F4A261",
    "Dati": "#264653",
    "Knafo": "#6D4C41",
    "Mariani": "#1D3557",
}

ID_TO_KEY = {
    "david_belliard": "Belliard",
    "emmanuel_gregoire": "Gregoire",
    "ian_brossat": "Brossat",
    "pierre_yves_bournazel": "Bournazel",
    "rachida_dati": "Dati",
    "sarah_knafo": "Knafo",
    "sophia_chikirou": "Chikirou",
    "thierry_mariani": "Mariani",
}

KEY_TO_LABEL = {
    "Belliard": "Belliard (EELV)",
    "Gregoire": "Grégoire (PS)",
    "Brossat": "Brossat (PCF)",
    "Bournazel": "Bournazel (Horizons)",
    "Dati": "Dati (LR)",
    "Knafo": "Knafo (Reconquête)",
    "Chikirou": "Chikirou (LFI)",
    "Mariani": "Mariani (RN)",
}

# Ordre idéologique gauche → droite
ORDRE_IDEO = [
    "Brossat",
    "Chikirou",
    "Belliard",
    "Gregoire",
    "Bournazel",
    "Dati",
    "Knafo",
    "Mariani",
]

# Palette topics (10 couleurs distinctes, non-rainbow)
TOPIC_COLORS = {
    "Logement & urbanisme": "#264653",
    "Vie locale & citoyennete": "#2A9D8F",
    "Geopolitique & international": "#E9C46A",
    "Social & solidarite": "#F4A261",
    "Securite & delinquance": "#E76F51",
    "Arrondissement / Liberte / Gauche": "#A8DADC",
    "Budget & finances": "#457B9D",
    "Education & culture": "#6D4C41",
    "Deux / Emmanuel / Peuple": "#BBBBBB",
    "Soutien / Jour / France": "#999999",
}

# Noms courts pour les topics (pour les axes)
TOPIC_SHORT = {
    "Logement & urbanisme": "Logement",
    "Vie locale & citoyennete": "Local/Paris",
    "Geopolitique & international": "Géopolitique",
    "Social & solidarite": "Social",
    "Securite & delinquance": "Sécurité",
    "Arrondissement / Liberte / Gauche": "Campagne gauche",
    "Budget & finances": "Budget",
    "Education & culture": "Anti-droite/Dati*",
    "Deux / Emmanuel / Peuple": "Générique†",
    "Soutien / Jour / France": "Soutien/France",
}

# ── Style matplotlib ──────────────────────────────────────────
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica Neue", "Helvetica", "Arial"],
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 10,
        "figure.dpi": 150,
        "savefig.dpi": 200,
        "savefig.bbox": "tight",
        "figure.facecolor": "white",
        "axes.facecolor": "white",
    }
)


def swiss_style(ax, title="", subtitle="", source="", grid_axis="y"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_color("#CCCCCC")
    ax.spines["bottom"].set_color("#CCCCCC")
    ax.tick_params(colors="#333333", labelsize=9, length=3)
    ax.set_facecolor("white")
    ax.figure.set_facecolor("white")
    if title:
        ax.set_title(
            title, fontsize=12, fontweight="bold", loc="left", color="#1a1a1a", pad=10
        )
    if subtitle:
        ax.text(
            0,
            1.03,
            subtitle,
            transform=ax.transAxes,
            fontsize=8.5,
            color="#666666",
            va="bottom",
        )
    if source:
        ax.text(
            1,
            -0.13,
            source,
            transform=ax.transAxes,
            fontsize=7.5,
            color="#999999",
            ha="right",
            va="top",
        )
    if grid_axis:
        ax.grid(axis=grid_axis, color="#F0F0F0", linewidth=0.5, zorder=0)
    return ax


print("✓ Cellule 0 OK — A2 Topics configuré")
print(f"  Topics palette : {len(TOPIC_COLORS)} topics")
print(f"  Output : {OUT}")

In [10]:
# ============================================================
# CELLULE 1 — Chargement données A2 + diagnostic LDA
# ============================================================

td = pd.read_csv(DATA_A2 / "topic_distribution.csv")
tr = pd.read_csv(DATA_A2 / "topic_resonance.csv")
ttl = pd.read_csv(DATA_A2 / "topic_timeline.csv", parse_dates=["week"])
twc = pd.read_csv(DATA_A2 / "topic_wordclouds_data.csv")

# Ajouter la clé candidat
td["key"] = td["candidate_id"].map(ID_TO_KEY)
tr["key"] = tr["candidate_id"].map(ID_TO_KEY)

print("=" * 60)
print("CHARGEMENT A2")
print("=" * 60)
for name, df in [
    ("topic_distribution", td),
    ("topic_resonance", tr),
    ("topic_timeline", ttl),
    ("topic_wordclouds_data", twc),
]:
    nans = df.isnull().sum().sum()
    print(f"  {name:<30} {df.shape}  NaN={nans}")

# ── Top-5 mots par topic ───────────────────────────────────────
print("\n" + "=" * 60)
print("TOPICS LDA — TOP 5 MOTS + DIAGNOSTIC")
print("=" * 60)
print(f"{'ID':<4} {'Topic name':<38} {'Top-5 mots':<50} {'Qualité'}")
print("-" * 110)

quality_flag = {
    "Logement & urbanisme": "✓ clair",
    "Vie locale & citoyennete": "✓ clair",
    "Geopolitique & international": "✓ clair",
    "Social & solidarite": "~ générique",
    "Securite & delinquance": "~ flou (mots: politique, service)",
    "Arrondissement / Liberte / Gauche": "✓ clair (gauche locale)",
    "Budget & finances": "✓ clair",
    "Education & culture": "⚠ mal nommé (mots: droite, dati, extrême)",
    "Deux / Emmanuel / Peuple": "⚠ fourre-tout (grégoire en 4e mot)",
    "Soutien / Jour / France": "~ générique",
}

topics_info = (
    twc.groupby(["topic_id", "topic_name"])
    .apply(lambda g: g.sort_values("rank")["word"].head(5).tolist())
    .reset_index(name="top5")
)
topics_info = topics_info.sort_values("topic_id")

for _, row in topics_info.iterrows():
    words = ", ".join(row["top5"])
    flag = quality_flag.get(row["topic_name"], "")
    print(f"  T{row['topic_id']:<3} {row['topic_name']:<38} {words:<50} {flag}")

# ── Note méthodologique ────────────────────────────────────────
print("\n" + "=" * 60)
print("NOTE MÉTHODOLOGIQUE")
print("=" * 60)
print("  k=10 pré-calculé — pas de score C_v disponible dans les données")
print("  → Cellule 2 : évaluation qualitative de la distribution (pas d'optimisation)")
print()
print("  ⚠ Renommages proposés :")
print("    T7 'Education & culture' → 'Anti-droite/Dati*'")
print("       (mots dominants: droite, dati, culture, extrême, fr)")
print("    T8 'Deux / Emmanuel / Peuple' → 'Générique†'")
print("       (mots dominants: deux, emmanuel, peuple, grégoire, européenne)")
print()
print("  ─ Ces 2 topics seront annotés avec * et † dans les figures")

# ── Stats engagement dans topic_resonance ─────────────────────
print("\n" + "=" * 60)
print("ENGAGEMENT PAR TOPIC (topic_resonance) — GLOBAL")
print("=" * 60)
eng_global = (
    tr.groupby("topic_name")
    .agg(
        n_tweets_total=("n_tweets", "sum"),
        med_engagement=("median_engagement", "median"),
        avg_engagement=("avg_engagement", "mean"),
    )
    .sort_values("med_engagement", ascending=False)
    .round(0)
)
print(eng_global.to_string())
print("\nNote : engagement = interactions absolues (likes+comments+shares), pas ER")

In [11]:
# ============================================================
# CELLULE 2 — Distribution des topics : concentration et diversité
#             (pas de multi-k disponible → évaluation qualitative)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Panel gauche : distribution des topic_pct (tous candidats) ─
ax_left = axes[0]

# Boxplot par topic : variabilité inter-candidats
topics_ordered = (
    td.groupby("topic_name")["topic_pct"]
    .mean()
    .sort_values(ascending=True)
    .index.tolist()
)

for i, t in enumerate(topics_ordered):
    vals = td[td["topic_name"] == t]["topic_pct"]
    med = vals.median()
    q1, q3 = vals.quantile(0.25), vals.quantile(0.75)
    lo = max(vals.min(), q1 - 1.5 * (q3 - q1))
    hi = min(vals.max(), q3 + 1.5 * (q3 - q1))
    col = TOPIC_COLORS.get(t, "#888888")

    ax_left.barh(i, q3 - q1, left=q1, height=0.55, color=col, alpha=0.8, zorder=3)
    ax_left.plot([lo, q1], [i, i], color=col, lw=1.3, zorder=3)
    ax_left.plot([q3, hi], [i, i], color=col, lw=1.3, zorder=3)
    ax_left.plot([med, med], [i - 0.28, i + 0.28], color="white", lw=2.2, zorder=4)
    ax_left.text(hi + 0.2, i, f"{med:.1f}%", va="center", fontsize=8, color="#333333")

ax_left.set_yticks(range(len(topics_ordered)))
ax_left.set_yticklabels([TOPIC_SHORT.get(t, t) for t in topics_ordered], fontsize=8.5)
ax_left.axvline(
    10, color="#999999", lw=0.8, ls="--", label="10% = répartition uniforme théorique"
)
ax_left.legend(fontsize=7.5, frameon=False, loc="lower right")
swiss_style(
    ax_left,
    title="Distribution des topics (variabilité inter-candidats)",
    subtitle="% du corpus par candidat — médiane + IQR sur 8 candidats",
    grid_axis="x",
)
ax_left.set_xlabel("% du corpus du candidat", fontsize=9)

# ── Panel droit : volume de tweets par topic (total corpus) ───
ax_right = axes[1]

vol = tr.groupby("topic_name")["n_tweets"].sum().reindex(topics_ordered)

colors_ordered = [TOPIC_COLORS.get(t, "#888888") for t in topics_ordered]
bars = ax_right.barh(
    range(len(topics_ordered)), vol.values, color=colors_ordered, alpha=0.85, height=0.6
)

for i, (t, v) in enumerate(zip(topics_ordered, vol.values)):
    ax_right.text(v + 20, i, f"{v:,}", va="center", fontsize=8, color="#333333")

ax_right.set_yticks(range(len(topics_ordered)))
ax_right.set_yticklabels([])
swiss_style(
    ax_right,
    title="Volume de tweets par topic",
    subtitle="Nombre total de tweets assignés à chaque topic (tous candidats)",
    source="Source : topic_resonance.csv · k=10 topics · LDA pré-calculé",
    grid_axis="x",
)
ax_right.set_xlabel("Nombre de tweets", fontsize=9)

plt.tight_layout()
plt.savefig(OUT / "A2_C2_distribution_topics.png", dpi=200, bbox_inches="tight")
plt.show()

print("✓ Figure sauvegardée")
print("\n" + "=" * 60)
print("DIAGNOSTIC DE QUALITÉ DES TOPICS")
print("=" * 60)

# Concentration : si un topic dépasse 15%, il sur-représente
print(f"\nTopics avec médiane > 12% (potentiel surpoids) :")
over = td.groupby("topic_name")["topic_pct"].median()
for t, v in over[over > 12].sort_values(ascending=False).items():
    print(f"  {TOPIC_SHORT.get(t,t):<25} médiane = {v:.1f}%")

print(f"\nTopics avec médiane < 8% (sous-représentés) :")
for t, v in over[over < 8].sort_values().items():
    print(f"  {TOPIC_SHORT.get(t,t):<25} médiane = {v:.1f}%")

# Herfindahl par candidat : plus c'est élevé, plus le discours est concentré
td["pct_sq"] = (td["topic_pct"] / 100) ** 2
hhi = td.groupby("key")["pct_sq"].sum().rename("HHI").round(4)
print(f"\nHerfindahl-Hirschman Index (HHI) — concentration discursive :")
print("  (HHI=0.1 = parfait équilibre entre 10 topics, HHI→1 = 1 topic dominant)")
for k in ORDRE_IDEO:
    if k in hhi.index:
        print(f"  {KEY_TO_LABEL[k]:<28} HHI = {hhi[k]:.4f}")

In [12]:
# ============================================================
# CELLULE 3 — Heatmap matrice candidat × topic
#             Clustering hiérarchique candidats + topics
# ============================================================

from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

# ── Pivot : candidats (lignes) × topics (colonnes) ───────────
pivot = td.pivot_table(index="key", columns="topic_name", values="topic_pct").reindex(
    index=ORDRE_IDEO
)

# Renommer les colonnes avec les noms courts
pivot.columns = [TOPIC_SHORT.get(c, c) for c in pivot.columns]
pivot.index = [KEY_TO_LABEL[k] for k in pivot.index]

# ── Linkages ──────────────────────────────────────────────────
link_rows = linkage(pdist(pivot.values, metric="euclidean"), method="ward")
link_cols = linkage(pdist(pivot.values.T, metric="euclidean"), method="ward")

# Ordre optimal issu du clustering
from scipy.cluster.hierarchy import leaves_list

row_order = leaves_list(link_rows)
col_order = leaves_list(link_cols)
pivot_reordered = pivot.iloc[row_order, col_order]

# ── Figure : heatmap + dendrogrammes ─────────────────────────
fig = plt.figure(figsize=(15, 8))
gs = fig.add_gridspec(
    2, 2, width_ratios=[0.15, 1], height_ratios=[0.2, 1], hspace=0.02, wspace=0.02
)

ax_dend_top = fig.add_subplot(gs[0, 1])
ax_dend_left = fig.add_subplot(gs[1, 0])
ax_heat = fig.add_subplot(gs[1, 1])

# Dendrogramme colonnes (topics)
dn_col = dendrogram(
    link_cols,
    ax=ax_dend_top,
    orientation="top",
    labels=pivot.columns[col_order].tolist(),
    color_threshold=0,
    above_threshold_color="#AAAAAA",
    leaf_font_size=0,
)
ax_dend_top.set_axis_off()

# Dendrogramme lignes (candidats)
dn_row = dendrogram(
    link_rows,
    ax=ax_dend_left,
    orientation="left",
    labels=pivot.index[row_order].tolist(),
    color_threshold=0,
    above_threshold_color="#AAAAAA",
    leaf_font_size=0,
)
ax_dend_left.set_axis_off()

# Heatmap
vals = pivot_reordered.values
im = ax_heat.imshow(
    vals, aspect="auto", cmap="YlOrRd", vmin=5, vmax=20, interpolation="nearest"
)

# Labels axes
ax_heat.set_xticks(range(len(pivot_reordered.columns)))
ax_heat.set_xticklabels(pivot_reordered.columns, rotation=35, ha="right", fontsize=8.5)
ax_heat.set_yticks(range(len(pivot_reordered.index)))
ax_heat.set_yticklabels(pivot_reordered.index, fontsize=9)

# Valeurs dans les cellules
for r in range(vals.shape[0]):
    for c in range(vals.shape[1]):
        v = vals[r, c]
        color = "white" if v > 15 else "#333333"
        ax_heat.text(
            c, r, f"{v:.1f}", ha="center", va="center", fontsize=7.5, color=color
        )

# Colorbar
cbar = plt.colorbar(im, ax=ax_heat, shrink=0.7, pad=0.01)
cbar.set_label("% du corpus candidat", fontsize=8)
cbar.ax.tick_params(labelsize=7.5)

# Couleurs des candidats sur l'axe Y
for i, label in enumerate(pivot_reordered.index):
    key = [k for k, v in KEY_TO_LABEL.items() if v == label][0]
    ax_heat.get_yticklabels()[i].set_color(COLORS[key])
    ax_heat.get_yticklabels()[i].set_fontweight("bold")

ax_heat.spines[:].set_visible(False)
ax_heat.tick_params(length=0)
ax_heat.set_facecolor("white")

fig.suptitle(
    "Matrice discursive — % du corpus par candidat × topic",
    fontsize=13,
    fontweight="bold",
    x=0.55,
    ha="center",
    y=1.01,
)
fig.text(
    0.98,
    0.0,
    "Source : topic_distribution.csv · k=10 topics LDA · Clustering hiérarchique Ward\n"
    "* T7 renommé (mots: droite, dati, extrême) · † T8 topic générique (grégoire en 4e mot)",
    ha="right",
    fontsize=7,
    color="#999999",
)

plt.savefig(OUT / "A2_C3_matrix_candidat_topic.png", dpi=200, bbox_inches="tight")
plt.show()
print("✓ Figure sauvegardée")

# ── Sortie texte ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("TOPICS DOMINANTS PAR CANDIDAT (top 3)")
print("=" * 60)
for key in ORDRE_IDEO:
    sub = td[td["key"] == key].sort_values("topic_pct", ascending=False)
    top3 = [
        (TOPIC_SHORT.get(r["topic_name"], r["topic_name"]), round(r["topic_pct"], 1))
        for _, r in sub.head(3).iterrows()
    ]
    label = KEY_TO_LABEL[key]
    print(
        f"  {label:<28} {top3[0][0]} ({top3[0][1]}%) · {top3[1][0]} ({top3[1][1]}%) · {top3[2][0]} ({top3[2][1]}%)"
    )

print("\n" + "=" * 60)
print("PROFIL D'ORIGINALITÉ — candidats les plus distinctifs")
print("  (écart max par rapport à la moyenne du corpus sur un topic)")
print("=" * 60)
mean_pct = td.groupby("topic_name")["topic_pct"].mean()
td["deviation"] = td.apply(lambda r: r["topic_pct"] - mean_pct[r["topic_name"]], axis=1)
for key in ORDRE_IDEO:
    sub = td[td["key"] == key].sort_values("deviation", ascending=False).iloc[0]
    print(
        f"  {KEY_TO_LABEL[key]:<28} +{sub['deviation']:.1f}% sur '{TOPIC_SHORT.get(sub['topic_name'], sub['topic_name'])}'"
    )

# Export pour A3
pivot.to_csv(OUT / "A2_matrix_candidat_topic.csv")
print(f"\n✓ Export matrice → {OUT / 'A2_matrix_candidat_topic.csv'}")

In [13]:
# ============================================================
# CELLULE 4 — Topics classés par engagement médian
#             Double axe : fréquence vs engagement
# ============================================================

# ── Agrégation : médiane des median_engagement sur 8 candidats ─
# On utilise median_engagement (robuste aux outliers) agrégé en médiane
# inter-candidats pour ne pas sur-pondérer Knafo/Dati qui ont plus de volume
eng_agg = (
    tr.groupby("topic_name")
    .agg(
        med_eng_global=("median_engagement", "median"),
        avg_eng_global=("avg_engagement", "mean"),
        n_tweets_total=("n_tweets", "sum"),
        n_candidats=("candidate_id", "nunique"),
    )
    .reset_index()
    .sort_values("med_eng_global", ascending=True)
)

eng_agg["topic_short"] = eng_agg["topic_name"].map(TOPIC_SHORT)
colors_bar = [TOPIC_COLORS.get(t, "#888") for t in eng_agg["topic_name"]]

fig, (ax_eng, ax_vol) = plt.subplots(1, 2, figsize=(16, 7), sharey=True)

# ── Barplot gauche : engagement médian ────────────────────────
bars = ax_eng.barh(
    range(len(eng_agg)),
    eng_agg["med_eng_global"],
    color=colors_bar,
    alpha=0.85,
    height=0.6,
)
for i, (_, row) in enumerate(eng_agg.iterrows()):
    ax_eng.text(
        row["med_eng_global"] + 20,
        i,
        f"{row['med_eng_global']:.0f}",
        va="center",
        fontsize=9,
        color="#333333",
        fontweight="bold",
    )

ax_eng.set_yticks(range(len(eng_agg)))
ax_eng.set_yticklabels(eng_agg["topic_short"], fontsize=9.5)

# Ligne de référence : médiane globale
med_global = eng_agg["med_eng_global"].median()
ax_eng.axvline(med_global, color="#999999", lw=1, ls="--", alpha=0.7)
ax_eng.text(
    med_global + 5,
    len(eng_agg) - 0.4,
    f"médiane\nglobale\n{med_global:.0f}",
    fontsize=7.5,
    color="#999",
    va="top",
)

swiss_style(
    ax_eng,
    title="Engagement médian par topic",
    subtitle="Médiane des median_engagement par candidat (interactions absolues : likes+comments+shares)",
    grid_axis="x",
)
ax_eng.set_xlabel("Interactions médianes (valeur absolue)", fontsize=9)

# ── Barplot droit : volume de tweets ──────────────────────────
# Réordonner selon le même tri (par engagement)
colors_bar2 = [TOPIC_COLORS.get(t, "#888") for t in eng_agg["topic_name"]]
ax_vol.barh(
    range(len(eng_agg)),
    eng_agg["n_tweets_total"],
    color=colors_bar2,
    alpha=0.5,
    height=0.6,
)
for i, (_, row) in enumerate(eng_agg.iterrows()):
    ax_vol.text(
        row["n_tweets_total"] + 5,
        i,
        f"{int(row['n_tweets_total'])}",
        va="center",
        fontsize=9,
        color="#555",
    )

swiss_style(
    ax_vol,
    title="Volume de tweets par topic",
    subtitle="Nombre total de tweets dans ce topic (tous candidats)",
    source="Source : topic_resonance.csv · Engagement = interactions absolues (non normalisé par followers)\n"
    "Note : topics T7* et T8† ont des noms originaux inadéquats (cf. Cellule 1)",
    grid_axis="x",
)
ax_vol.set_xlabel("Nombre de tweets", fontsize=9)

# Annoter la divergence engagement vs volume
# Trouver le topic avec le plus haut engagement et celui avec le plus grand volume
top_eng = eng_agg.iloc[-1]["topic_short"]
top_vol = eng_agg.loc[eng_agg["n_tweets_total"].idxmax(), "topic_short"]
if top_eng != top_vol:
    fig.text(
        0.5,
        -0.03,
        f'⟶ Topic le plus engageant : "{top_eng}" · Topic le plus fréquent : "{top_vol}"'
        " — ils ne coïncident pas (cf. Mersmann & Werne 2026)",
        ha="center",
        fontsize=9,
        color="#555",
        style="italic",
    )

plt.tight_layout()
plt.savefig(OUT / "A2_C4_topic_engagement.png", dpi=200, bbox_inches="tight")
plt.show()

# ── Sortie texte ──────────────────────────────────────────────
print("✓ Figure sauvegardée")
print("\n" + "=" * 60)
print("TOPICS CLASSÉS PAR ENGAGEMENT MÉDIAN (décroissant)")
print("=" * 60)
print(f"{'Topic':<28} {'Med.eng':>8} {'N tweets':>10} {'Candidats':>10}")
print("-" * 60)
for _, row in eng_agg.sort_values("med_eng_global", ascending=False).iterrows():
    print(
        f"  {row['topic_short']:<26} {row['med_eng_global']:>8.0f} {int(row['n_tweets_total']):>10} {int(row['n_candidats']):>10}"
    )

top_vol_name = eng_agg.loc[eng_agg["n_tweets_total"].idxmax(), "topic_short"]
top_eng_name = eng_agg.sort_values("med_eng_global").iloc[-1]["topic_short"]
print(f"\n→ Topic le plus fréquent : {top_vol_name}")
print(f"→ Topic le plus engageant : {top_eng_name}")
print(f"→ Coïncident : {top_vol_name == top_eng_name}")
print("\n⚠ Note : engagement en valeurs absolues — candidats avec grandes audiences")
print("  (Knafo, Dati) pèsent plus dans avg_engagement que dans median_engagement.")
print("  La médiane inter-candidats est plus robuste pour la comparaison.")

In [14]:
# ============================================================
# CELLULE 5 — Timeline des topics : stacked area + anomalies A1
# ============================================================

import matplotlib.dates as mdates

# ── Chargement des anomalies A1 ───────────────────────────────
ano_path = DATA_A1_OUT / "A1_anomalies_for_A2_A3.csv"
if ano_path.exists():
    ano = pd.read_csv(ano_path, parse_dates=["week_start"])
    print(f"✓ Anomalies A1 chargées : {len(ano)} lignes")
else:
    ano = None
    print("⚠ Fichier anomalies A1 non trouvé — run A1 Cellule 7 d'abord")

# ── Pivot timeline : semaine × topic ──────────────────────────
ttl["week"] = pd.to_datetime(ttl["week"])
ttl_filtered = ttl[ttl["week"] >= "2025-01-01"].copy()

pivot_tl = ttl_filtered.pivot_table(
    index="week", columns="topic_name", values="n_tweets", aggfunc="sum", fill_value=0
)
# Renommer colonnes
pivot_tl.columns = [TOPIC_SHORT.get(c, c) for c in pivot_tl.columns]

# Ordre des topics : par volume total décroissant (les plus fréquents en bas)
topic_order_vol = pivot_tl.sum(axis=0).sort_values(ascending=True).index.tolist()
pivot_tl = pivot_tl[topic_order_vol]

colors_tl = []
for t_short in topic_order_vol:
    # Retrouver la couleur via le nom original
    t_orig = next((k for k, v in TOPIC_SHORT.items() if v == t_short), None)
    colors_tl.append(TOPIC_COLORS.get(t_orig, "#888888") if t_orig else "#888888")

# ── Figure ────────────────────────────────────────────────────
fig, (ax_top, ax_bot) = plt.subplots(
    2,
    1,
    figsize=(16, 9),
    gridspec_kw={"height_ratios": [3, 1.2], "hspace": 0.08},
    sharex=True,
)

# Stacked area
bottom = np.zeros(len(pivot_tl))
for topic, color in zip(topic_order_vol, colors_tl):
    vals = pivot_tl[topic].values
    ax_top.fill_between(
        pivot_tl.index,
        bottom,
        bottom + vals,
        color=color,
        alpha=0.82,
        linewidth=0,
        label=topic,
    )
    # Label direct sur la bande si volume suffisant
    mean_val = vals.mean()
    if mean_val > 3:
        idx_mid = len(vals) // 2
        mid_y = bottom[idx_mid] + vals[idx_mid] / 2
        ax_top.text(
            pivot_tl.index[idx_mid],
            mid_y,
            topic,
            fontsize=7,
            ha="center",
            va="center",
            color="white",
            fontweight="bold",
            alpha=0.9,
        )
    bottom += vals

# Anomalies A1 : lignes verticales
if ano is not None:
    for _, row in ano.iterrows():
        if row["week_start"] >= pd.Timestamp("2025-01-01"):
            ax_top.axvline(
                row["week_start"], color="#222222", lw=0.9, ls="--", alpha=0.5, zorder=5
            )
            ax_top.text(
                row["week_start"],
                ax_top.get_ylim()[1] * 0.97,
                row["key"][:3] + ".",
                fontsize=7,
                color="#222222",
                ha="center",
                va="top",
                fontweight="bold",
            )

swiss_style(
    ax_top,
    title="Évolution temporelle des topics — volume hebdomadaire (janv. 2025 – fév. 2026)",
    subtitle="Tweets par semaine par topic (tous candidats confondus) · Pointillés = anomalies A1",
    grid_axis="y",
)
ax_top.set_ylabel("Nbre de tweets / semaine", fontsize=9)
ax_top.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x)}"))

# ── Panel bas : avg_eng par topic (ligne) ─────────────────────
pivot_eng = ttl_filtered.pivot_table(
    index="week", columns="topic_name", values="avg_eng", aggfunc="median", fill_value=0
)
pivot_eng.columns = [TOPIC_SHORT.get(c, c) for c in pivot_eng.columns]

# Ne tracer que les 3 topics les plus engageants
top3_eng = (
    eng_agg.sort_values("med_eng_global", ascending=False)
    .head(3)["topic_short"]
    .tolist()
)
for t in top3_eng:
    if t in pivot_eng.columns:
        t_orig = next((k for k, v in TOPIC_SHORT.items() if v == t), None)
        c = TOPIC_COLORS.get(t_orig, "#888") if t_orig else "#888"
        series = pivot_eng[t].rolling(3, center=True).median()
        ax_bot.plot(pivot_eng.index, series, color=c, lw=1.8, label=t)

swiss_style(
    ax_bot,
    title="",
    subtitle="",
    source="Source : topic_timeline.csv · N=591 semaines×topics · Rolling médiane 3 semaines pour les 3 topics les plus engageants",
    grid_axis="y",
)
ax_bot.set_ylabel("Eng. médian\n(rolling 3w)", fontsize=8)
ax_bot.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k" if x >= 1000 else str(int(x)))
)
ax_bot.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax_bot.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax_bot.xaxis.get_majorticklabels(), rotation=30, ha="right", fontsize=8)
ax_bot.legend(fontsize=7.5, frameon=False, loc="upper left", ncol=3)

plt.savefig(OUT / "A2_C5_timeline_topics.png", dpi=200, bbox_inches="tight")
plt.show()

print("✓ Figure sauvegardée")
print("\n" + "=" * 60)
print("VOLUME MOYEN PAR TOPIC PAR SEMAINE")
print("=" * 60)
vol_weekly = pivot_tl.mean(axis=0).sort_values(ascending=False)
for t, v in vol_weekly.items():
    print(f"  {t:<28} {v:.1f} tweets/semaine en moyenne")

In [15]:
# ============================================================
# CELLULE 6 — Croisement A2 × A1 : quel topic pendant chaque anomalie ?
#             + Export CSV pour A3
# ============================================================

if ano is None:
    print("⚠ Anomalies A1 non disponibles — skip")
else:
    # ── Pour chaque anomalie, trouver le topic dominant cette semaine-là ─
    results = []
    for _, a_row in ano.iterrows():
        week = a_row["week_start"]
        # Fenêtre ±3 jours pour matcher la semaine du topic_timeline
        mask = (ttl["week"] >= week - pd.Timedelta(days=5)) & (
            ttl["week"] <= week + pd.Timedelta(days=5)
        )
        sub = ttl[mask]
        if len(sub) == 0:
            # Essai avec fenêtre ±10 jours
            mask2 = (ttl["week"] >= week - pd.Timedelta(days=12)) & (
                ttl["week"] <= week + pd.Timedelta(days=12)
            )
            sub = ttl[mask2]

        if len(sub) > 0:
            # Topic dominant = celui avec le plus haut avg_eng cette semaine
            top_eng_row = sub.loc[sub["avg_eng"].idxmax()]
            top_vol_row = sub.loc[sub["n_tweets"].idxmax()]
            results.append(
                {
                    "semaine": week.strftime("%d %b %Y"),
                    "candidat": a_row["key"],
                    "z_global": round(a_row["z_global"], 2),
                    "top_engagement": int(a_row.get("top_post_engagement", 0)),
                    "topic_eng_max": TOPIC_SHORT.get(
                        top_eng_row["topic_name"], top_eng_row["topic_name"]
                    ),
                    "avg_eng_semaine": round(top_eng_row["avg_eng"], 0),
                    "topic_vol_max": TOPIC_SHORT.get(
                        top_vol_row["topic_name"], top_vol_row["topic_name"]
                    ),
                    "n_tweets_semaine": int(top_vol_row["n_tweets"]),
                    "convergence": top_eng_row["topic_name"]
                    == top_vol_row["topic_name"],
                }
            )
        else:
            results.append(
                {
                    "semaine": week.strftime("%d %b %Y"),
                    "candidat": a_row["key"],
                    "z_global": round(a_row["z_global"], 2),
                    "topic_eng_max": "N/A",
                    "topic_vol_max": "N/A",
                    "convergence": False,
                }
            )

    cross_df = pd.DataFrame(results).sort_values("z_global", ascending=False)

    # ── Figure : tableau visuel ──────────────────────────────
    fig, ax = plt.subplots(figsize=(16, 8))
    ax.axis("off")

    cols_show = [
        "semaine",
        "candidat",
        "z_global",
        "topic_eng_max",
        "topic_vol_max",
        "convergence",
    ]
    col_labels = [
        "Semaine",
        "Candidat",
        "Z-score",
        "Topic le + engageant",
        "Topic le + fréquent",
        "Convergent ?",
    ]
    table_data = cross_df[cols_show].values

    tbl = ax.table(
        cellText=table_data,
        colLabels=col_labels,
        cellLoc="left",
        loc="center",
        bbox=[0, 0, 1, 1],
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8.5)
    for (row, col), cell in tbl.get_celld().items():
        cell.set_edgecolor("#E0E0E0")
        if row == 0:
            cell.set_facecolor("#1a1a1a")
            cell.set_text_props(color="white", fontweight="bold")
        elif col == 1 and row > 0:
            cand = table_data[row - 1][1]
            cell.set_facecolor(COLORS.get(str(cand), "#FFFFFF") + "33")
        elif col == 5 and row > 0:
            conv = table_data[row - 1][5]
            cell.set_facecolor("#D4EDDA" if conv else "#FFF3CD")
        else:
            cell.set_facecolor("#FFFFFF" if row % 2 == 0 else "#F8F8F8")

    ax.set_title(
        "Croisement A2 × A1 — Topic dominant lors de chaque anomalie temporelle",
        fontsize=12,
        fontweight="bold",
        loc="left",
        pad=10,
        color="#1a1a1a",
    )
    fig.text(
        0.98,
        0.01,
        "Source : topic_timeline.csv × A1_anomalies_for_A2_A3.csv · "
        "Topic engageant = avg_eng max · Topic fréquent = n_tweets max · "
        "Vert = convergence",
        ha="right",
        fontsize=7.5,
        color="#999999",
    )

    plt.savefig(OUT / "A2_C6_cross_A1.png", dpi=200, bbox_inches="tight")
    plt.show()

    # ── Sortie texte ─────────────────────────────────────────
    print("✓ Figure sauvegardée")
    print("\n" + "=" * 65)
    print("CROISEMENT ANOMALIES × TOPICS")
    print("=" * 65)
    n_conv = cross_df["convergence"].sum()
    print(
        f"  Anomalies avec convergence eng/vol sur même topic : {n_conv}/{len(cross_df)}"
    )
    print(f"\n  Topics les plus fréquents lors des anomalies :")
    for t, n in cross_df["topic_eng_max"].value_counts().items():
        print(f"    {t:<28} : {n} anomalie(s)")
    print(
        f"\n  Question narrative : les anomalies sont-elles thématiques ou événementielles ?"
    )
    n_diff = (cross_df["topic_eng_max"] != cross_df["topic_vol_max"]).sum()
    print(f"  Anomalies où topic engageant ≠ topic fréquent : {n_diff}/{len(cross_df)}")
    print(
        f"  → Si n_diff élevé : la viralité n'est pas le topic le plus discuté, mais un sujet de niche"
    )

    # ── Exports ──────────────────────────────────────────────
    cross_df.to_csv(OUT / "A2_cross_anomalies_topics.csv", index=False)

    # Export global pour A3
    eng_agg[["topic_name", "topic_short", "med_eng_global", "n_tweets_total"]].to_csv(
        OUT / "A2_topic_engagement_ranking.csv", index=False
    )
    pivot.to_csv(OUT / "A2_matrix_candidat_topic.csv")

    print(f"\n✓ Exports A2 :")
    print(f"   → {OUT/'A2_cross_anomalies_topics.csv'}  (utilisé par A3, A7)")
    print(f"   → {OUT/'A2_topic_engagement_ranking.csv'} (utilisé par A3)")
    print(f"   → {OUT/'A2_matrix_candidat_topic.csv'}    (utilisé par A3, A7)")
    print("\n✓ A2 TERMINÉ — 4 figures + 3 CSV exportés")

# A2 — Résonance Thématique (LDA)
**Paris Municipales 2026 — Pipeline v3**

Objectifs :
- Identifier les thèmes des tweets de chaque candidat
- Mesurer la résonance (engagement) par thème
- Évolution temporelle des sujets
- Convergence/divergence thématique entre candidats

In [16]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

BASE  = Path("..") / ".."  # repo root when run from final/A2_topics/
DATA  = BASE / "final" / "data"

CANDIDATE_COLORS = {
    "sarah_knafo":           "#1B2A4A",
    "rachida_dati":          "#0066CC",
    "thierry_mariani":       "#0D1B2A",
    "sophia_chikirou":       "#CC0000",
    "emmanuel_gregoire":     "#FF6699",
    "ian_brossat":           "#DD0000",
    "david_belliard":        "#00A86B",
    "pierre_yves_bournazel": "#FF8C00",
}
SHORT = {
    "sarah_knafo": "Knafo", "rachida_dati": "Dati", "thierry_mariani": "Mariani",
    "sophia_chikirou": "Chikirou", "emmanuel_gregoire": "Gregoire",
    "ian_brossat": "Brossat", "david_belliard": "Belliard",
    "pierre_yves_bournazel": "Bournazel",
}
TEMPLATE = "plotly_dark"

A2_DATA = BASE / "final" / "A2_topics" / "data"

tweets   = pd.read_csv(A2_DATA / "tweets_for_topics.csv", encoding="utf-8-sig")
dist     = pd.read_csv(A2_DATA / "topic_distribution.csv", encoding="utf-8-sig")
res      = pd.read_csv(A2_DATA / "topic_resonance.csv", encoding="utf-8-sig")
timeline = pd.read_csv(A2_DATA / "topic_timeline.csv", encoding="utf-8-sig")
kw       = pd.read_csv(A2_DATA / "topic_wordclouds_data.csv", encoding="utf-8-sig")

tweets["timestamp"] = pd.to_datetime(tweets["timestamp"], utc=True, errors="coerce")
timeline["week"]    = pd.to_datetime(timeline["week"], errors="coerce")

print(f"Tweets : {len(tweets)} | Themes : {dist['topic_name'].nunique()}")
print("\nThèmes identifiés :")
for t in kw.drop_duplicates('topic_id').itertuples():
    top = kw[kw.topic_id == t.topic_id]['word'].head(6).tolist()
    print(f"  T{t.topic_id+1}: {t.topic_name} | {', '.join(top)}")


## 1. Re-entraînement LDA (optionnel — pour explorer d'autres n_topics)

In [ ]:
# OPTIONNEL : relancer le LDA avec d'autres parametres
# from sklearn.decomposition import LatentDirichletAllocation
# from sklearn.feature_extraction.text import CountVectorizer
# import re
# def clean(text):
#     text = re.sub(r"http\S+|@\w+", " ", str(text))
#     return re.sub(r"[^\w\s]", " ", text).lower()
# corpus = tweets["text"].apply(clean)
# vec = CountVectorizer(min_df=3, max_df=0.85, ngram_range=(1,2), max_features=2000)
# X = vec.fit_transform(corpus)
# lda = LatentDirichletAllocation(n_components=12, random_state=42)
# doc_topics = lda.fit_transform(X)
print("LDA pré-calculé disponible dans dist et res. Décommenter ci-dessus pour re-entraîner.")


## 2. Heatmap distribution thématique

In [ ]:
pivot = dist.pivot_table(index="label", columns="topic_name", values="topic_pct", aggfunc="sum", fill_value=0)

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=list(pivot.columns),
    y=list(pivot.index),
    colorscale="RdYlGn", zmin=0, zmax=30,
    text=[[f"{v:.1f}%" for v in row] for row in pivot.values],
    texttemplate="%{text}",
    hovertemplate="<b>%{y}</b><br>%{x}: %{z:.1f}%<extra></extra>",
    colorbar=dict(title="% corpus"),
))
fig.update_layout(
    template=TEMPLATE,
    title="Distribution thématique par candidat",
    xaxis=dict(tickangle=-35),
    width=1050, height=500,
)
fig.show()


## 3. Résonance : engagement moyen par thème

In [ ]:
fig = px.scatter(
    res, x="n_tweets", y="avg_engagement",
    size="total_engagement", color="label",
    hover_name="topic_name",
    template=TEMPLATE,
    title="Volume vs Engagement moyen par thème et candidat",
    labels={"avg_engagement": "Engagement moyen", "n_tweets": "N tweets"},
)
fig.show()


## 4. Évolution temporelle des thèmes

In [ ]:
pivot_t = timeline.pivot_table(index="topic_name", columns="week", values="n_tweets", aggfunc="sum", fill_value=0)
pivot_t.columns = [str(c)[:10] for c in pivot_t.columns]

fig = go.Figure(go.Heatmap(
    z=pivot_t.values, x=list(pivot_t.columns), y=list(pivot_t.index),
    colorscale="Blues",
    hovertemplate="Theme: %{y}<br>Semaine: %{x}<br>N: %{z}<extra></extra>",
))
fig.update_layout(
    template=TEMPLATE, title="Évolution temporelle des thèmes",
    xaxis=dict(tickangle=-45), width=1100, height=500,
)
fig.show()
